# News Intelligence Studio — Main Notebook

**Team 116:** Chaman Kumar, Satyam Priyam, Killol

This is the primary notebook entry point for the project. It executes the full project from the bundled raw datasets in `data/raw/`, saves all outputs to `results/`, displays benchmark tables and figures, and lets you enter a custom news article for both analysis and verification.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 1. Confirm the bundled raw datasets are present

In [ ]:
required_raw_files = [
    PROJECT_ROOT / 'data' / 'raw' / 'kaggle_fake_real' / 'Fake.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'kaggle_fake_real' / 'True.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'fakenewsnet' / 'BuzzFeed_fake_news_content.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'fakenewsnet' / 'BuzzFeed_real_news_content.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'fakenewsnet' / 'PolitiFact_fake_news_content.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'fakenewsnet' / 'PolitiFact_real_news_content.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'news_category' / 'News_Category_Dataset_v3.json',
]

missing = [str(p.relative_to(PROJECT_ROOT)) for p in required_raw_files if not p.exists()]
print('Missing files:' if missing else 'All required raw data files are present.')
missing

## 2. Full project execution from raw data

This cell calls `demo.run_project(...)`, which is the single standalone project runner. It rebuilds the analyzer from `data/raw/`, refreshes saved outputs in `results/`, and then runs analysis plus verification on one article.

In [ ]:
from demo import run_project

project_run = run_project(PROJECT_ROOT, n_topics=12, extra_real_rows=2000, topic_rows=12000)
project_run['rebuild_summary']

## 3. Load and display the saved outputs

In [ ]:
from src.notebook_runner import load_saved_outputs
from IPython.display import display

outputs = load_saved_outputs(PROJECT_ROOT)
display(outputs['classifier_benchmark'])
display(outputs['topic_benchmark'])
display(outputs['dataset_split'])
display(outputs['source_breakdown'])
outputs['classification_metrics']['test_metrics']

## 4. Display saved result figures

In [ ]:
from IPython.display import Image, display

for image_name in ['class_balance.png', 'confusion_matrix.png', 'topic_distribution.png', 'top_subjects.png']:
    image_path = PROJECT_ROOT / 'results' / 'figures' / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path), width=700))

## 5. Enter your own news article

Edit the headline and article text below, then run the next two code cells. The combined analyzer + verifier output will be shown in the notebook and saved to `results/custom_article_result.json`.

In [ ]:
custom_headline = "Iran's ability to threaten Strait of Hormuz is degraded, US says"
custom_article_text = """
IDF says missiles launched from Iran towards Israel. In the last few minutes,
the Israel Defense Forces said it had identified missiles launched towards the country.
Defensive systems were operating to intercept the threat. A precautionary alert was sent
to people in relevant areas.
"""

In [ ]:
from src.notebook_runner import analyze_and_save_custom_article
from IPython.display import Markdown, JSON

custom_result = analyze_and_save_custom_article(PROJECT_ROOT, custom_headline, custom_article_text)

print('Analyzer prediction:', custom_result['analysis']['prediction'])
print('Analyzer confidence:', round(custom_result['analysis']['confidence'], 4))
print('Closest theme:', custom_result['analysis']['topic']['topic_name'])
print('Summary:', custom_result['analysis']['summary'])
print('Verification summary:', custom_result['verification']['summary'])

JSON(custom_result)

## 6. Optional standalone run

The same full execution path is available from the terminal with:

```bash
python demo.py
```